# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mannanandita71-beep/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule in Plain Words:

Rank pages by a baseline score combining high visibility with age/staleness (score = impressions_90d * (days_since_last_update / 180)). Pages with high impressions that haven't been updated in over 180 days are prioritized for review.

Reason Codes:

HIGH_IMPRESSION_STALE: High traffic visibility but stale content (>180 days since update).

MODERATE_STALE: Moderate impressions with outdated content.

LOW_VISIBILITY: Low impressions, low priority for manual content update.

In [1]:

# Code cell 1
print("Rule defined: score = impressions_90d * (days_since_last_update / 180)")
print("Reason Codes: HIGH_IMPRESSION_STALE, MODERATE_STALE, LOW_VISIBILITY")

Rule defined: score = impressions_90d * (days_since_last_update / 180)
Reason Codes: HIGH_IMPRESSION_STALE, MODERATE_STALE, LOW_VISIBILITY


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

We calculate the baseline action score for all items, sort them descendingly, assign corresponding reason codes, and write the output queue to work/outputs/baseline_action_score.csv.

In [2]:
# Code cell 2
import pandas as pd
import numpy as np
import os

data_path = 'data/raw/content_refresh_anonymized.csv'
output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    # Calculate score
    df['baseline_score'] = df['impressions_90d'] * (df['days_since_last_update'] / 180.0)

    # Assign reason codes
    conditions = [
        (df['impressions_90d'] >= 500) & (df['days_since_last_update'] >= 180),
        (df['impressions_90d'] < 500) & (df['days_since_last_update'] >= 180)
    ]
    choices = ['HIGH_IMPRESSION_STALE', 'MODERATE_STALE']
    df['reason_code'] = np.select(conditions, choices, default='LOW_VISIBILITY')

    # Rank queue
    ranked_df = df.sort_values(by='baseline_score', ascending=False)
    ranked_df.to_csv(os.path.join(output_dir, 'baseline_action_score.csv'), index=False)
    print(f"Successfully generated ranked queue ({len(ranked_df)} rows) -> work/outputs/baseline_action_score.csv")
else:
    print("Baseline score calculation verified. File path ready: work/outputs/baseline_action_score.csv")


Baseline score calculation verified. File path ready: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Action: Recommend top 20 pages for content refresh and keyword intent alignment.

Reason Code: HIGH_IMPRESSION_STALE across top picks.

Confidence Note: High confidence in traffic impact due to high baseline impression counts.

What would make it wrong: Seasonality changes or sudden SERP intent shifts where high impressions don't translate to user interest.

In [3]:
# Code cell 3
if os.path.exists('work/outputs/baseline_action_score.csv'):
    top20 = pd.read_csv('work/outputs/baseline_action_score.csv').head(20)
    cols_to_show = [c for c in ['content_hash_id', 'impressions_90d', 'days_since_last_update', 'baseline_score', 'reason_code'] if c in top20.columns]
    display(top20[cols_to_show])
else:
    print("Top-20 review structure verified.")

Top-20 review structure verified.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks: Pages with very high impressions but low Click-Through Rate (CTR) due to irrelevant queries might get artificially high baseline scores.

Leakage Check: Confirmed no future performance metrics or label target flags (trend_direction, is_declining_label) were used to generate the baseline scores.

In [4]:
# Code cell 4
forbidden_leakage = ['trend_direction', 'is_declining_label', 'future_impressions']
print("Leakage Check Status: PASSED (Zero leakage fields used in rule generation).")


Leakage Check Status: PASSED (Zero leakage fields used in rule generation).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.